# Module 12 — Capstone: pgvector RAG backend

Prereqs:
- `pgvector` installed on your Postgres 17 server (see README).
- `12-capstone/schema.sql` already loaded:
  ```powershell
  psql -U postgres -d pgcourse -f 12-capstone/schema.sql
  ```
- `pip install -r requirements.txt` (includes `sentence-transformers`, `pgvector`).

Flow: load corpus → embed → store → hybrid retrieve → tune.

In [ ]:
import os, hashlib
from dotenv import load_dotenv
load_dotenv()
DSN = f"postgresql://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}@{os.environ['PGHOST']}:{os.environ['PGPORT']}/{os.environ['PGDATABASE']}"
print('OK')

## 1. Tiny corpus

Ten Postgres-flavoured paragraphs is enough to demonstrate every retrieval path. Replace with your own docs to scale.

In [ ]:
CORPUS = [
    ('doc://btree',   'B-Tree indexes',
        'A B-Tree index in PostgreSQL supports equality and range queries. The default index type. It is balanced and ordered, so ORDER BY queries can use it without an explicit sort.'),
    ('doc://gin',     'GIN indexes',
        'GIN indexes are inverted indexes good for many-valued columns: jsonb, arrays, and tsvector. They are slower to build and larger than B-Tree but excellent for containment and full-text search.'),
    ('doc://gist',    'GiST indexes',
        'GiST is a framework for building index types on custom data, including geometric types, ranges, and trigrams. It supports nearest-neighbour search via the distance operator.'),
    ('doc://brin',    'BRIN indexes',
        'BRIN indexes summarise blocks of pages and are ideal for very large, naturally ordered tables such as time-series. They use a tiny fraction of the space of a B-Tree.'),
    ('doc://mvcc',    'MVCC',
        'PostgreSQL uses multi-version concurrency control. Readers do not block writers and writers do not block readers. Each transaction sees a consistent snapshot of the database.'),
    ('doc://wal',     'Write-Ahead Log',
        'The WAL is an append-only log of every change. It is the foundation of durability, streaming replication, point-in-time recovery, and logical decoding.'),
    ('doc://vacuum',  'VACUUM',
        'VACUUM reclaims storage from dead tuples left by UPDATE and DELETE. Autovacuum runs it automatically. VACUUM FULL rewrites the table and locks readers.'),
    ('doc://hnsw',    'HNSW indexes (pgvector)',
        'HNSW is a graph-based approximate nearest-neighbour index in pgvector. It is fast and accurate but uses more memory than IVFFlat. Tune ef_search at query time for recall.'),
    ('doc://rls',     'Row-Level Security',
        'Row-Level Security policies restrict which rows a role can read or modify. Combined with a per-request SET LOCAL setting, it implements multi-tenancy in a single database.'),
    ('doc://replica', 'Streaming replication',
        'Streaming replication ships WAL from a primary to one or more standbys. Synchronous replication waits for standby confirmation before commit; asynchronous does not.'),
]
print(len(CORPUS), 'docs')

## 2. Embed with a small CPU model

`all-MiniLM-L6-v2` is 384-d, fast, and good enough for a demo. First run downloads ~80 MB.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
DIM = 384

def embed(texts):
    return model.encode(texts, normalize_embeddings=True).tolist()

print(len(embed(['hello world'])[0]), 'dim')

## 3. Ingest

Idempotent: re-running this cell with the same corpus does nothing because `source_uri` is a unique key and `content_sha` matches.

Note: for simplicity each document is one chunk here. A real pipeline would split long docs into ~200-300 token windows with overlap.

In [ ]:
import psycopg
from pgvector.psycopg import register_vector

def sha(text): return hashlib.sha256(text.encode('utf-8')).digest()

with psycopg.connect(DSN) as conn:
    register_vector(conn)
    with conn.cursor() as cur:
        for source_uri, title, body in CORPUS:
            content_sha = sha(body)
            # Insert / update document; detect 'unchanged'
            cur.execute(
                """INSERT INTO rag.documents (source_uri, title, metadata, content_sha)
                   VALUES (%s, %s, %s::jsonb, %s)
                   ON CONFLICT (source_uri) DO UPDATE
                     SET title = EXCLUDED.title,
                         updated_at = now(),
                         content_sha = EXCLUDED.content_sha
                   RETURNING id,
                             (rag.documents.content_sha = EXCLUDED.content_sha) AS unchanged
                """,
                (source_uri, title, '{"tenant":"demo"}', content_sha),
            )
            doc_id, unchanged = cur.fetchone()
            if unchanged:
                continue
            # Replace chunks atomically
            cur.execute("DELETE FROM rag.chunks WHERE document_id = %s", (doc_id,))
            emb = embed([body])[0]
            cur.execute(
                "INSERT INTO rag.chunks (document_id, ordinal, text, token_count, embedding) VALUES (%s, %s, %s, %s, %s)",
                (doc_id, 0, body, len(body.split()), emb),
            )
    conn.commit()
    with conn.cursor() as cur:
        cur.execute('SELECT count(*) FROM rag.documents')
        print('documents:', cur.fetchone()[0])
        cur.execute('SELECT count(*) FROM rag.chunks')
        print('chunks:', cur.fetchone()[0])

## 4. Pure vector search

In [ ]:
def vector_search(q_text, k=5):
    qv = embed([q_text])[0]
    with psycopg.connect(DSN) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            cur.execute(
                """SELECT c.id, d.title, c.text,
                          1 - (c.embedding <=> %s::vector) AS similarity
                   FROM rag.chunks c
                   JOIN rag.documents d ON d.id = c.document_id
                   ORDER BY c.embedding <=> %s::vector
                   LIMIT %s""",
                (qv, qv, k),
            )
            return cur.fetchall()

for row in vector_search('how do I make a query plan faster on a big table'):
    print(round(row[3], 3), '\t', row[1])

## 5. FTS only

In [ ]:
def fts_search(q_text, k=5):
    with psycopg.connect(DSN) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """SELECT c.id, d.title, c.text,
                          ts_rank(c.fts, q) AS rank
                   FROM rag.chunks c
                   JOIN rag.documents d ON d.id = c.document_id,
                        websearch_to_tsquery('english', %s) q
                   WHERE c.fts @@ q
                   ORDER BY rank DESC
                   LIMIT %s""",
                (q_text, k),
            )
            return cur.fetchall()

for row in fts_search('vacuum dead tuples bloat'):
    print(round(row[3], 3), '\t', row[1])

## 6. Hybrid retrieval with RRF

In [ ]:
def hybrid(q_text, k=5, n=50, rrf_k=60):
    qv = embed([q_text])[0]
    with psycopg.connect(DSN) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            cur.execute(
                """WITH vec AS (
                       SELECT id, row_number() OVER (ORDER BY embedding <=> %s::vector) AS r
                       FROM rag.chunks ORDER BY embedding <=> %s::vector LIMIT %s
                   ),
                   lex AS (
                       SELECT c.id, row_number() OVER (ORDER BY ts_rank(c.fts, q) DESC) AS r
                       FROM rag.chunks c, websearch_to_tsquery('english', %s) q
                       WHERE c.fts @@ q LIMIT %s
                   )
                   SELECT c.id, d.title, c.text,
                          coalesce(1.0/(%s + v.r), 0) + coalesce(1.0/(%s + l.r), 0) AS rrf
                   FROM rag.chunks c
                   JOIN rag.documents d ON d.id = c.document_id
                   LEFT JOIN vec v ON v.id = c.id
                   LEFT JOIN lex l ON l.id = c.id
                   WHERE v.id IS NOT NULL OR l.id IS NOT NULL
                   ORDER BY rrf DESC
                   LIMIT %s""",
                (qv, qv, n, q_text, n, rrf_k, rrf_k, k),
            )
            return cur.fetchall()

for row in hybrid('how do I make slow analytics queries on big tables faster'):
    print(round(row[3], 4), '\t', row[1])

## 7. Tune HNSW recall vs latency

`ef_search` is the search-time knob. Try 10, 40, 100, 200 and compare which neighbours come back.

In [ ]:
import time

def time_vector(query, ef):
    qv = embed([query])[0]
    with psycopg.connect(DSN) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            cur.execute('SET LOCAL hnsw.ef_search = %s', (ef,))
            t0 = time.perf_counter()
            cur.execute(
                """SELECT id FROM rag.chunks ORDER BY embedding <=> %s::vector LIMIT 5""",
                (qv,),
            )
            ids = [r[0] for r in cur.fetchall()]
            ms = (time.perf_counter() - t0) * 1000
            return ms, ids

for ef in [10, 40, 100, 200]:
    ms, ids = time_vector('how does write-ahead logging work', ef)
    print(f'ef_search={ef:>3}  {ms:6.2f} ms  -> {ids}')

## Done

What you built:
- A `rag.documents` + `rag.chunks` schema with HNSW + FTS indexes.
- An idempotent ingest pipeline keyed on `source_uri` and `content_sha`.
- Three retrievers: vector, FTS, hybrid (RRF). All composable from Python.
- A way to dial recall vs latency at query time.

From here:
- Chunk long documents with overlap.
- Add a per-tenant filter to every query.
- Re-rank the top ~50 hybrid candidates with a cross-encoder (`sentence-transformers/ms-marco-MiniLM-L-6-v2`).
- Quantize embeddings to `halfvec` once you have enough rows to feel the storage cost.